In [1]:
# Install dependencies
# !nvidia-smi
!pip install ultralytics torch torchvision --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 950.0/950.0 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 94.5 MB/s eta 0:00:00


In [11]:
from google.colab import files
from google.colab import drive
import xml.etree.ElementTree as ET
import os
import random
import torch
from tqdm import tqdm
import zipfile
from ultralytics import YOLO

In [3]:
USE_PARTIAL_DATA = True       # Enable partial data training
PARTIAL_DATA_RATIO = 0.3      # Use 30% of the dataset
DRIVE_CACHING = True          # Enable Google Drive caching
AUTO_DEVICE = True            # Auto-select GPU/CPU

YOLO_CONFIG_TEMPLATE = """
path: {dataset_path}
train: images/train
val: images/val
names:
{classes}
"""

# Path configurations
DRIVE_ROOT = '/content/drive/MyDrive/YOLO_VOC2012'
LOCAL_WORKSPACE = '/content/voc_workspace'

In [4]:
# Global constants
CLASS_NAMES = [
  'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat',
  'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person',
  'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

# Device configuration
device = 'cuda' if torch.cuda.is_available() and AUTO_DEVICE else 'cpu'

In [5]:
def create_partial_dataset():
  """Create sampled dataset with specified ratio"""
  print(f"Creating partial dataset ({PARTIAL_DATA_RATIO*100}% sampling)...")
  partial_dir = os.path.join(LOCAL_WORKSPACE, "partial_data")

  # Create directory structure
  for split in ['train', 'val']:
    os.makedirs(os.path.join(partial_dir, "images", split), exist_ok=True)
    os.makedirs(os.path.join(partial_dir, "labels", split), exist_ok=True)

  # Helper function for safe symlinking
  def safe_symlink(src, dst):
    try:
      if not os.path.exists(src):
        raise FileNotFoundError(f"Source path does not exist: {src}")
      os.makedirs(os.path.dirname(dst), exist_ok=True)
      if not os.path.exists(dst):
        os.symlink(src, dst)
        print(f"Created symlink: {dst} -> {src}")
      return True
    except Exception as e:
      print(f"Symlink creation failed: {str(e)}")
      return False

  # Process both splits
  for split in ['train', 'val']:
    src_img_dir = os.path.join(LOCAL_WORKSPACE, "datasets", "images", split)
    all_images = os.listdir(src_img_dir)
    sampled_images = random.sample(all_images, int(len(all_images)*PARTIAL_DATA_RATIO))

    for img_file in sampled_images:
      base_name = os.path.splitext(img_file)[0]

      # Process image
      src_img = os.path.join(src_img_dir, img_file)
      dst_img = os.path.join(partial_dir, "images", split, img_file)
      safe_symlink(src_img, dst_img)

      # Process label
      src_label = os.path.join(LOCAL_WORKSPACE, "datasets", "labels", split, f"{base_name}.txt")
      dst_label = os.path.join(partial_dir, "labels", split, f"{base_name}.txt")
      if os.path.exists(src_label):
        safe_symlink(src_label, dst_label)

  if DRIVE_CACHING:
    !tar -czf {partial_dir}.tar.gz -C {partial_dir} .
    !cp {partial_dir}.tar.gz {DRIVE_ROOT}/partial_data/

In [6]:
def process_raw_data():
  """Convert raw VOC2012 annotations to YOLO format"""
  # Ensure the workspace directory exists
  os.makedirs(LOCAL_WORKSPACE, exist_ok=True)

  # Download the dataset
  voc_tar_path = f"{LOCAL_WORKSPACE}/VOC2012.tar"
  if not os.path.exists(voc_tar_path):
    print("Downloading VOC2012 dataset...")
    !wget -q -O {voc_tar_path} http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar
    print(f"Downloaded size: {os.path.getsize(voc_tar_path)} bytes")

  # Extract the dataset
  print("Extracting dataset...")
  !tar -xf {voc_tar_path} -C {LOCAL_WORKSPACE} 2>&1 | tee extraction.log
  if "error" in open("extraction.log").read().lower():
    raise RuntimeError("Dataset extraction failed")

  # Validate the extraction results
  required_dirs = [
    "VOCdevkit/VOC2012/Annotations",
    "VOCdevkit/VOC2012/JPEGImages",
    "VOCdevkit/VOC2012/ImageSets/Main"
  ]
  for dir_path in required_dirs:
    if not os.path.exists(os.path.join(LOCAL_WORKSPACE, dir_path)):
      raise FileNotFoundError(f"Missing critical directory: {dir_path}")

  # Create YOLO directory structure (with multi-level directory creation)
  voc_root = os.path.join(LOCAL_WORKSPACE, "VOCdevkit/VOC2012")
  yolo_dir = os.path.join(LOCAL_WORKSPACE, "datasets")

  # Create YOLO directory structure
  for split in ['train', 'val']:
    os.makedirs(os.path.join(yolo_dir, "images", split), exist_ok=True)
    os.makedirs(os.path.join(yolo_dir, "labels", split), exist_ok=True)

  # Load train/val filenames (read from ImageSets/Main)
  def load_split_names(split='train'):
    split_file = os.path.join(voc_root, "ImageSets/Main", f"{split}.txt")
    with open(split_file, 'r') as f:
      return [line.strip() for line in f.readlines()]

  # Parse XML annotation
  def convert_annotation(image_id, split):
    in_xml = os.path.join(voc_root, "Annotations", f"{image_id}.xml")
    out_txt = os.path.join(yolo_dir, "labels", split, f"{image_id}.txt")

    tree = ET.parse(in_xml)
    root = tree.getroot()

    size = root.find('size')
    img_width = int(size.find('width').text)
    img_height = int(size.find('height').text)

    with open(out_txt, 'w') as f:
      for obj in root.iter('object'):
        cls_name = obj.find('name').text
        if cls_name not in CLASS_NAMES:
          continue  # Skip undefined classes
        cls_id = CLASS_NAMES.index(cls_name)

        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)

        # Convert to YOLO format (normalized coordinates)
        x_center = (xmin + xmax) / 2 / img_width
        y_center = (ymin + ymax) / 2 / img_height
        width = (xmax - xmin) / img_width
        height = (ymax - ymin) / img_height

        f.write(f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

    # Create symbolic link for image file
    src_img = os.path.join(voc_root, "JPEGImages", f"{image_id}.jpg")
    dst_img = os.path.join(yolo_dir, "images", split, f"{image_id}.jpg")
    if os.path.exists(src_img) and not os.path.exists(dst_img):
      os.symlink(src_img, dst_img)

  # Process training and validation sets
  for split in ['train', 'val']:
    image_ids = load_split_names(split)
    print(f"Processing {split} set ({len(image_ids)} images)...")
    for image_id in tqdm(image_ids):
      convert_annotation(image_id, split)

  print("Dataset conversion completed!")

In [7]:
"""Dataset preparation workflow"""
print("\n=== Preparing Dataset ===")

os.makedirs(LOCAL_WORKSPACE, exist_ok=True)

# Mount Google Drive if caching enabled
if DRIVE_CACHING:
  drive.mount('/content/drive', force_remount=True)
  os.makedirs(f"{DRIVE_ROOT}/processed_data", exist_ok=True)
  os.makedirs(f"{DRIVE_ROOT}/partial_data", exist_ok=True)

# Check for cached data
if DRIVE_CACHING and os.path.exists(f"{DRIVE_ROOT}/processed_data/.success"):
  print("Attempting to load cached data...")
  !rsync -avh --progress {DRIVE_ROOT}/processed_data/ {LOCAL_WORKSPACE}/datasets/
  if not os.path.exists(f"{LOCAL_WORKSPACE}/datasets/images/train"):
    print("Cached data incomplete, reprocessing...")
    process_raw_data()
else:
  print("Processing raw data...")
  process_raw_data()
  if DRIVE_CACHING:
    print("Archiving processed data...")
    !tar -czvf {LOCAL_WORKSPACE}/processed_data.tar.gz -C {LOCAL_WORKSPACE}/datasets .
    !cp -v {LOCAL_WORKSPACE}/processed_data.tar.gz {DRIVE_ROOT}/processed_data/
    !touch {DRIVE_ROOT}/processed_data/.success

if USE_PARTIAL_DATA:
  print("Creating partial dataset...")
  create_partial_dataset()

required_paths = [
  os.path.join(LOCAL_WORKSPACE, "datasets/images/train"),
  os.path.join(LOCAL_WORKSPACE, "datasets/labels/train")
]
for path in required_paths:
  if not os.path.exists(path):
    raise FileNotFoundError(f"Critical path missing: {path}")
  print(f"Verified existence: {path}")


=== Preparing Dataset ===
Mounted at /content/drive
Attempting to load cached data...
sending incremental file list
created directory /content/voc_workspace/datasets
./
.success
              0 100%    0.00kB/s    0:00:00 (xfr#1, to-chk=1/3)
processed_data.tar.gz
        779.81K 100%   20.95MB/s    0:00:00 (xfr#2, to-chk=0/3)

sent 780.18K bytes  received 111 bytes  520.19K bytes/sec
total size is 779.81K  speedup is 1.00
Cached data incomplete, reprocessing...
Downloaded size: 1999639040 bytes
Extracting dataset...
Processing train set (5717 images)...


100%|██████████| 5717/5717 [00:01<00:00, 4668.08it/s]


Processing val set (5823 images)...


100%|██████████| 5823/5823 [00:01<00:00, 4905.54it/s]


流式输出内容被截断，只能显示最后 5000 行内容。
Created symlink: /content/voc_workspace/partial_data/images/train/2008_008150.jpg -> /content/voc_workspace/datasets/images/train/2008_008150.jpg
Created symlink: /content/voc_workspace/partial_data/labels/train/2008_008150.txt -> /content/voc_workspace/datasets/labels/train/2008_008150.txt
Created symlink: /content/voc_workspace/partial_data/images/train/2009_000804.jpg -> /content/voc_workspace/datasets/images/train/2009_000804.jpg
Created symlink: /content/voc_workspace/partial_data/labels/train/2009_000804.txt -> /content/voc_workspace/datasets/labels/train/2009_000804.txt
Created symlink: /content/voc_workspace/partial_data/images/train/2010_005101.jpg -> /content/voc_workspace/datasets/images/train/2010_005101.jpg
Created symlink: /content/voc_workspace/partial_data/labels/train/2010_005101.txt -> /content/voc_workspace/datasets/labels/train/2010_005101.txt
Created symlink: /content/voc_workspace/partial_data/images/train/2010_001550.jpg -> /content/voc

In [8]:
def create_yolo_config(dataset_path, class_names):
  """Generate YOLO config file using template"""
  classes_str = '\n'.join([f'  {i}: {cls}' for i, cls in enumerate(class_names)])
  return YOLO_CONFIG_TEMPLATE.format(dataset_path=dataset_path,
                    classes=classes_str).strip()

def train_model():
  """Main training workflow"""
  print("\n=== Starting Training ===")

  # Select dataset path
  dataset_path = os.path.join(LOCAL_WORKSPACE, "partial_data" if USE_PARTIAL_DATA else "datasets")

  # Create YOLO config
  config_path = os.path.join(dataset_path, "voc_config.yaml")
  config_content = create_yolo_config(dataset_path, CLASS_NAMES)
  with open(config_path, 'w') as f:
    f.write(config_content)

  # Initialize model
  model = YOLO('yolo11n.pt').to(device)

  # Training parameters
  training_params = {
    'data': config_path,
    'epochs': 50 if USE_PARTIAL_DATA else 100,
    'batch': 32 if USE_PARTIAL_DATA else 16,
    'imgsz': 416 if USE_PARTIAL_DATA else 640,
    'device': device,
    'optimizer': 'Adam',
    'lr0': 0.01,
    'amp': True,
    'cache': 'disk' if DRIVE_CACHING else 'ram',
    'project': 'voc2012_train',
    'name': 'exp'
  }

  # Start training
  results = model.train(**training_params)
  print(f"Training completed! Results saved to: {results.save_dir}")

In [9]:
train_model()


=== Starting Training ===


100%|██████████| 5.35M/5.35M [00:00<00:00, 99.3MB/s]


Ultralytics 8.3.98 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolo11n.pt, data=/content/voc_workspace/partial_data/voc_config.yaml, epochs=50, time=None, patience=100, batch=32, imgsz=416, save=True, save_period=-1, cache=disk, device=cuda, workers=8, project=voc2012_train, name=exp, exist_ok=False, pretrained=True, optimizer=Adam, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=T

100%|██████████| 755k/755k [00:00<00:00, 26.3MB/s]


Overriding model.yaml nc=80 with nc=20

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytic

...

                   all       1746       4743      0.416      0.346      0.319      0.198

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      50/50      3.24G      1.207      1.692      1.382         47        416: 100%|██████████| 54/54 [00:15<00:00,  3.60it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 28/28 [00:08<00:00,  3.12it/s]


                   all       1746       4743      0.435      0.346      0.324      0.204

50 epochs completed in 0.423 hours.
Optimizer stripped from voc2012_train/exp/weights/last.pt, 5.4MB
Optimizer stripped from voc2012_train/exp/weights/best.pt, 5.4MB

Validating voc2012_train/exp/weights/best.pt...
Ultralytics 8.3.98 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLO11n summary (fused): 100 layers, 2,586,052 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 28/28 [00:10<00:00,  2.58it/s]


                   all       1746       4743      0.432      0.346      0.324      0.204
             aeroplane        102        142      0.579      0.577      0.574      0.379
               bicycle         82        103      0.343      0.476      0.376      0.258
                  bird        121        200      0.312      0.285       0.18     0.0909
                  boat         75        131      0.276      0.168      0.153      0.105
                bottle        102        218      0.388      0.119      0.116     0.0614
                   bus         60        108      0.762      0.537      0.618      0.493
                   car        181        352      0.658      0.403      0.448        0.3
                   cat        181        203      0.524      0.516      0.467      0.286
                 chair        186        439      0.362      0.128      0.168     0.0767
                   cow         39         90      0.199      0.265      0.107     0.0653
           diningtabl

In [14]:
# Save results
def zipdir(path, ziph):
  for root, dirs, files in os.walk(path):
    for file in files:
      file_path = os.path.join(root, file)
      ziph.write(file_path, os.path.relpath(file_path, os.path.join(path, '..')))
with zipfile.ZipFile('training_results.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
  zipdir('voc2012_train/exp/', zipf)
files.download('training_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>